# Statistical regularization Experiments
You have seen in the previous notebook that image reconstruction is an ill-defined problem.<br>
With regularization (i.e.: Solve $\arg\min_x \lVert Ax-y \rVert + \beta f\left(x\right) $, with $f (x)$ representing the "solution prior probability") you can achieve a denoised and unique solution in an easy way.  You can always see this as optimizing the *data fidelity term* plus the *regularization term* (or *prior probability* term)

The simplest $f(x)$ is the spatial gradient of the image: suppress strong variations from pixel to pixel: $\lVert \nabla x \rVert^2$ <br>
Its gradient (for the iterative reconstruction update) is: $\nabla^T \nabla x$. <br>
You can compute the spatial gradient using finite differences, like using `np.diff` and the `axis` parameter to differentiate along x or y

Do the following:
1. Try denoising on the deblurring problem, using a difficult condition(e.g.: $\sigma=2,\;\varepsilon=0.02$). Notice that there is no theoretical optimal value of $\beta$, try manually multiple values
2. Implement denoising for MR reconstruction in an undersampled case.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
i1 = np.load('testMR.npy')

# Task 1: Implement the *regularized* iterative for deblurring
The equations and the concepts are the same of the previous notebook. Now $\mathcal{L}=\lVert Ax-y \rVert + \beta f\left(x\right)$<br>
The gradient descent iterative optimization algorithm works the same way, but the gradient becomes the sum of two terms:<br>
$g=A^Tr + +\beta \nabla f(x)=A^T r + \beta \nabla \cdot \nabla x$

So, the previous algorithm becomes: 
Set $x^0=0$; $y^0=0$. The "residual" $r=y-Ax$ is initialized as y. The iterative algorithm repeats the following:
1. The gradient is $g = A^T r + \beta \nabla \cdot \nabla x$ 
2. Compute the optimal step size: $\alpha= \frac{||g||^2}{||Ag||^2 + \beta \lVert \nabla g \rVert^2}$
3. Update $x$ with $x^{k+1}=x^k+\alpha g$
4. Update the residual $r^{k+1}=y-Ax^{k+1}$. Since $A$ is linear you can avoid to compute a second time $A$. Indeed $r^{k+1}=y-A^{k}+Ag^k = r^k + \alpha Ag$, so you can reuse $\alpha g$ from the previous step

Details:
* You can monitor your loss by just computing: $\left\lVert r \right \rVert^2 $, after every iteration. Remember that the vast majority of the time is spent computing $Ax$ or $A^T y$. Doing $\left\lVert r \right \rVert^2 $ is very cheap
* Write a function, or a class, that does the reconstruction. Parameters that the user need to set are: starting image (0 or a different initial guess), number of iterations, measured data. This function should return/provide: reconstructed image, loss value.
* In this problem $A$ if the convolution with a gaussian, and $A^T$ is the same operation. Write the previous function in a way that it's trivial to adapt it to all other cases (Radon transform for CT, FFT for MR)

# Task 2: Write some undersampling schemes for MR and apply denoising reconstruction

In this case $A$ is `np.fft.fft2` and $A^T$ is `np.fft.ifft2`. Notice that for FFT $A^T = A^{-1}$, which means that in an iterative algorithm with exact step size you'll converge in one iteration!!

**Notice:** the FFT as implemented by default in `numpy` is such that `ifft` is not the exact transpose of `fft`, because of the "normalization". In the default: $FFT:\; A_k=\sum_{i\in [0,N]} a_m \,exp \left(-2\pi i mk/N\right) $ and  $ IFFT:\; a_m=\frac{1}{N} \sum_{k\in [0,N]} A_k\, exp \left(2\pi i mk/N\right) $

The factor $\frac{1}{N}$ if present only in `ifft`. Set `np.fft.fft2(norm='ortho')` and `np.fft.ifft2(norm='ortho')` so that it applies a factor $\frac{1}{\sqrt{N}}$ in both functions. This way $A^T$ and $A$ are perfectly the transpose. Otherwise the step size computation will be wrong!

## Task 2.1
In a real-world MR the simplest thing to do to speed up an acquisition is to skip acquiring some line (whole line!) in k-space in the phase encoding direction.<br>
But is this optimal? And in case, how would you do this? Uniform probability? Different probabilities? Are there some frequencies you absolutely must retain?<br>
Notice that it would be optimal to randomly skip individual *points* in k-space, rather than lines. Compare how images at the same MSE from the true "look", if there's some pattern in the noise.

**How do you reconstruct them analytically?**
For a start, assume that the missing frequencies are zero (it's your best guess!) and take the inverse fourier transform.

The next level will be doing an iterative reconstruction where you know which frequencies have not been acquired. Say you have this mask $M$. <br>
Your iterative reconstruction algorithm will be:<br>
$\lVert M\left[\mathcal{F}x -y \right] \rVert^2$<br>
You are not computing the MSE on frequencies you have not acquired!! Let the iterative reconstruction guess them!

Also here add the regularization term and see how effective it is compared to different acquisition mechanisms!